In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
# 1. Install the unrar utility into the Colab environment
!sudo apt-get install unrar -y

# 2. Create your meaningful target folder
!mkdir -p raw_seed_images

# 3. Extract the RAR file directly into the target folder
# (Using 'x' to extract with full directory structure preserved)
!unrar x "/content/drive/My Drive/VeriCure_Project/Dataset_v2.rar" raw_seed_images/

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/drive/My Drive/VeriCure_Project/Dataset_v2.rar


Would you like to replace the existing file raw_seed_images/Dataset_v2/froben/box001_back.jpg
178494 bytes, modified on 2026-07-13 06:21
with a new one
178494 bytes, modified on 2026-07-13 06:21

[Y]es, [N]o, [A]ll, n[E]ver, [R]ename, [Q]uit [Y]es

[Y]es, [N]o, [A]ll, n[E]ver, [R]ename, [Q]uit Y

Extracting  raw_seed_images/Dataset_v2/froben/box001_back.jpg              0%  OK 

Would you like to replace the existing file raw_seed_images/Dataset_v2/froben/box001_front.jpg
218884 bytes, modified on 2026-07-13 06:21
with a new one
218884 bytes, modified on 2026-07-13 06:21

[Y]es, [N]o, [A]ll, n[E]ver, [R]ename, [Q]u

In [14]:
!ls raw_seed_images/Dataset_v2

froben	glucophage  nise  panadol


In [15]:
import os
import random
import cv2
import numpy as np

# =====================================================================
# GLOBAL CONFIGURATIONS & RESOLUTION TUNING
# =====================================================================
# Point this directly to the nested folder layer where the medicines live
RAW_SOURCE_DIR = "raw_seed_images/Dataset_v2"
DATASET_DIR = "augmented_final"

# Verified folder names from your RAR extraction file structure
CLASSES = ["froben", "glucophage", "nise", "panadol"]
TARGET_COUNT = 500

# Standardized resolution to optimize storage footprint and avoid OOM errors
TARGET_SIZE = (512, 512)

def create_directory_structure():
    """Ensures distinct output folders exist for all medicine categories."""
    if not os.path.exists(DATASET_DIR):
        os.makedirs(DATASET_DIR)

    for c in CLASSES:
        for status in ["_Genuine", "_Counterfeit"]:
            path = os.path.join(DATASET_DIR, f"{c}{status}")
            if not os.path.exists(path):
                os.makedirs(path)
                print(f"[+] Created directory: {path}")

def load_source_images(folder_path):
    """Loads and rescales all seed images within a specific target class."""
    images = []
    if not os.path.exists(folder_path):
        print(f"[!] Warning: Folder path not found: {folder_path}")
        return images

    for file in os.listdir(folder_path):
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(folder_path, file)
            img = cv2.imread(img_path)
            if img is not None:
                # Force immediate area relation resizing to save RAM allocation overhead
                img_resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
                images.append(img_resized)
    return images

# ==========================================
# CORE AUGMENTATION TRANSFORMS
# ==========================================

def apply_geometric_tweaks(image):
    """Simulates multi-angle phone scanning alignments and hand jitters."""
    h, w = image.shape[:2]

    # Minor rotation boundaries between -8 and +8 degrees
    angle = random.uniform(-8, 8)
    matrix = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    rotated = cv2.warpAffine(image, matrix, (w, h), borderMode=cv2.BORDER_REPLICATE)

    # Minor translation shifts to handle off-center alignment variations
    shift_x = random.randint(-15, 15)
    shift_y = random.randint(-15, 15)
    translation_matrix = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
    shifted = cv2.warpAffine(rotated, translation_matrix, (w, h), borderMode=cv2.BORDER_REPLICATE)

    return shifted

def tweak_lighting(image):
    """Simulates illumination variance inside stores and homes."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    # Quantize the Value channel to force brightness fluctuations
    v_factor = random.uniform(0.85, 1.15)
    v = np.clip(v * v_factor, 0, 255).astype(np.uint8)

    merged = cv2.merge([h, s, v])
    return cv2.cvtColor(merged, cv2.COLOR_HSV2BGR)

def inject_counterfeit_anomalies(image):
    """Simulates printing defects, inferior ink chemistry, and plate shifts."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    # 1. Shift Hue slightly to mimic off-brand dye mixtures
    hue_shift = random.randint(-6, 6)
    h = ((h.astype(int) + hue_shift) % 180).astype(np.uint8)

    # 2. Alter Saturation parameters to simulate low-grade print quality
    sat_factor = random.uniform(0.8, 1.25)
    s = np.clip(s * sat_factor, 0, 255).astype(np.uint8)

    modified_hsv = cv2.merge([h, s, v])
    bgr_img = cv2.cvtColor(modified_hsv, cv2.COLOR_HSV2BGR)

    # 3. Apply a fine Gaussian blur to reproduce low-quality typography bleeding
    if random.random() > 0.5:
        bgr_img = cv2.GaussianBlur(bgr_img, (3, 3), 0)

    # 4. Introduce minor contrast alterations
    alpha = random.uniform(0.9, 1.1)
    beta = random.randint(-10, 10)
    bgr_img = cv2.convertScaleAbs(bgr_img, alpha=alpha, beta=beta)

    return bgr_img

# ==========================================
# MAIN DATASET GENERATION PIPELINE
# ==========================================

def generate_dataset():
    create_directory_structure()

    # High-quality target JPEG quantization matrix to maintain file sizes under 1GB
    encode_params = [int(cv2.IMWRITE_JPEG_QUALITY), 85]

    for c in CLASSES:
        current_source_dir = os.path.join(RAW_SOURCE_DIR, c)
        base_images = load_source_images(current_source_dir)

        if not base_images:
            print(f"[!] Skipping category '{c}': No valid source images detected.")
            continue

        print(f"\n[*] Processing '{c}': Loaded {len(base_images)} base cropped images.")

        # 1. Execute Genuine Data Splitting Loop
        genuine_out_dir = os.path.join(DATASET_DIR, f"{c}_Genuine")
        for count in range(TARGET_COUNT):
            base_img = random.choice(base_images)
            augmented = apply_geometric_tweaks(base_img)
            augmented = tweak_lighting(augmented)

            out_path = os.path.join(genuine_out_dir, f"{c}_genuine_{count}.jpg")
            cv2.imwrite(out_path, augmented, encode_params)

        # 2. Execute Counterfeit Data Splitting Loop
        counterfeit_out_dir = os.path.join(DATASET_DIR, f"{c}_Counterfeit")
        for count in range(TARGET_COUNT):
            base_img = random.choice(base_images)
            augmented = apply_geometric_tweaks(base_img)
            augmented = tweak_lighting(augmented)
            augmented = inject_counterfeit_anomalies(augmented)

            out_path = os.path.join(counterfeit_out_dir, f"{c}_counterfeit_{count}.jpg")
            cv2.imwrite(out_path, augmented, encode_params)

        print(f"[+] Successfully structured 1,000 balanced vectors for class: {c}")

    print("\n=======================================================================")
    print("[SUCCESS] Data engineering phase complete. Dataset is fully optimized.")
    print("=======================================================================")

if __name__ == "__main__":
    generate_dataset()


[*] Processing 'froben': Loaded 58 base cropped images.
[+] Successfully structured 1,000 balanced vectors for class: froben

[*] Processing 'glucophage': Loaded 80 base cropped images.
[+] Successfully structured 1,000 balanced vectors for class: glucophage

[*] Processing 'nise': Loaded 80 base cropped images.
[+] Successfully structured 1,000 balanced vectors for class: nise

[*] Processing 'panadol': Loaded 80 base cropped images.
[+] Successfully structured 1,000 balanced vectors for class: panadol

[SUCCESS] Data engineering phase complete. Dataset is fully optimized.


In [16]:
# 1. Compress the entire augmented folder into a single size-optimized zip file
!zip -r augmented_final.zip augmented_final/

# 2. Copy the zip archive straight into your permanent Google Drive project folder
!cp augmented_final.zip "/content/drive/My Drive/VeriCure_Project/"

print("\n=======================================================================")
print("[SUCCESS] Backup complete! 'augmented_final.zip' is safely stored in Drive.")
print("=======================================================================")

  adding: augmented_final/ (stored 0%)
  adding: augmented_final/panadol_Counterfeit/ (stored 0%)
  adding: augmented_final/panadol_Counterfeit/panadol_counterfeit_468.jpg (deflated 0%)
  adding: augmented_final/panadol_Counterfeit/panadol_counterfeit_237.jpg (deflated 0%)
  adding: augmented_final/panadol_Counterfeit/panadol_counterfeit_448.jpg (deflated 0%)
  adding: augmented_final/panadol_Counterfeit/panadol_counterfeit_109.jpg (deflated 0%)
  adding: augmented_final/panadol_Counterfeit/panadol_counterfeit_403.jpg (deflated 0%)
  adding: augmented_final/panadol_Counterfeit/panadol_counterfeit_120.jpg (deflated 0%)
  adding: augmented_final/panadol_Counterfeit/panadol_counterfeit_86.jpg (deflated 0%)
  adding: augmented_final/panadol_Counterfeit/panadol_counterfeit_405.jpg (deflated 0%)
  adding: augmented_final/panadol_Counterfeit/panadol_counterfeit_342.jpg (deflated 0%)
  adding: augmented_final/panadol_Counterfeit/panadol_counterfeit_363.jpg (deflated 0%)
  adding: augmented_fin

In [17]:
# 1. Install unrar utility if not already present
!sudo apt-get install unrar -y

# 2. Create the temporary local input folder
!mkdir -p raw_unknown_seeds

# 3. Extract your custom background archive automatically, forcing overwrites
!unrar x -o+ "/content/drive/My Drive/VeriCure_Project/unknown.rar" raw_unknown_seeds/

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/drive/My Drive/VeriCure_Project/unknown.rar

Creating    raw_unknown_seeds/unknown                                 OK
Extracting  raw_unknown_seeds/unknown/unknown_001.jpeg                     1%  OK 
Extracting  raw_unknown_seeds/unknown/unknown_002.jpeg                     2%  OK 
Extracting  raw_unknown_seeds/unknown/unknown_003.jpeg                     3%  OK 
Extracting  raw_unknown_seeds/unknown/unknown_004.jpeg                     4%  OK 
Extracting  raw_unknown_seeds/unknown/unknown_005.jpeg                     5%  OK 
Extracting  raw_unknown_seeds/unknown/unknown_006.jpeg                     6%  OK 
Extracting

In [18]:
import os
import random
import cv2
import numpy as np

# =====================================================================
# GLOBAL CONFIGURATIONS
# =====================================================================
# Point this to your newly extracted custom unknown folder path
RAW_UNKNOWN_DIR = "raw_unknown_seeds"
DATASET_DIR = "augmented_final"
UNKNOWN_OUT_DIR = os.path.join(DATASET_DIR, "Background_Unknown")

TARGET_COUNT = 500
TARGET_SIZE = (512, 512)

def setup_environment():
    """Ensures the validation class output directory exists."""
    if not os.path.exists(UNKNOWN_OUT_DIR):
        os.makedirs(UNKNOWN_OUT_DIR)
        print(f"[+] Created target directory: {UNKNOWN_OUT_DIR}")

def load_custom_seeds(folder_path):
    """Recursively crawls and pre-resizes your 100 custom background photos."""
    images = []
    if not os.path.exists(folder_path):
        print(f"[!] Critical Error: Root seed path not found: {folder_path}")
        return images

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, file)
                img = cv2.imread(img_path)
                if img is not None:
                    # Resize instantly to save system memory allocation limits
                    img_resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
                    images.append(img_resized)
    return images

# ==========================================
# DEFOCUS & ENVIRONMENTAL NOISE ENGINE
# ==========================================

def apply_defocus_blur(image):
    """Simulates a camera lens dropping focus bounds on background elements."""
    if random.random() > 0.4:
        kernel_size = random.choice([5, 7, 9, 11, 15])
        return cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)
    return image

def inject_sensor_grain(image):
    """Simulates multi-ISO camera noise configurations."""
    if random.random() > 0.5:
        row, col, ch = image.shape
        mean = 0
        sigma = random.uniform(12, 28)
        gauss = np.random.normal(mean, sigma, (row, col, ch))
        noisy = image.astype(np.float32) + gauss
        return np.clip(noisy, 0, 255).astype(np.uint8)
    return image

def extreme_lighting_shift(image):
    """Varies value exposures to simulate fluorescent lights and shadows."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    v_factor = random.uniform(0.5, 1.3)
    v = np.clip(v * v_factor, 0, 255).astype(np.uint8)

    merged = cv2.merge([h, s, v])
    return cv2.cvtColor(merged, cv2.COLOR_HSV2BGR)

# ==========================================
# MAIN ROUTINE PIPELINE
# ==========================================

def generate_custom_unknowns():
    setup_environment()

    print("[*] Loading custom unknown seed matrix from disk...")
    base_seeds = load_custom_seeds(RAW_UNKNOWN_DIR)

    if not base_seeds:
        print("[!] Execution Halted: No valid background frames found. Check path nesting.")
        return

    print(f"[+] Loaded {len(base_seeds)} base custom background images successfully.")
    print(f"[*] Scaling vectors up to match target size distribution ({TARGET_COUNT} files)...")

    encode_params = [int(cv2.IMWRITE_JPEG_QUALITY), 85]

    for count in range(TARGET_COUNT):
        # Pick a random image from your 100 custom background seeds
        base_img = random.choice(base_seeds)

        # Apply random environment and focus variations
        processed = extreme_lighting_shift(base_img)
        processed = apply_defocus_blur(processed)
        processed = inject_sensor_grain(processed)

        # Output directly to your final dataset folder matrix
        out_path = os.path.join(UNKNOWN_OUT_DIR, f"unknown_bg_{count}.jpg")
        cv2.imwrite(out_path, processed, encode_params)

    print("\n=======================================================================")
    print(f"[SUCCESS] Custom Background_Unknown generation complete: {TARGET_COUNT} files.")
    print("=======================================================================")

if __name__ == "__main__":
    generate_custom_unknowns()

[+] Created target directory: augmented_final/Background_Unknown
[*] Loading custom unknown seed matrix from disk...
[+] Loaded 100 base custom background images successfully.
[*] Scaling vectors up to match target size distribution (500 files)...

[SUCCESS] Custom Background_Unknown generation complete: 500 files.


In [19]:
import os
import shutil

# =====================================================================
# PATH CONFIGURATIONS
# =====================================================================
SOURCE_DIR = "augmented_final"
STAGING_DIR = "dataset_complete"
DRIVE_DESTINATION = "/content/drive/My Drive/VeriCure_Project/"

# The exact 9-class structural roadmap required by your model pipeline
EXPECTED_CLASSES = [
    "froben_Genuine", "froben_Counterfeit",
    "glucophage_Genuine", "glucophage_Counterfeit",
    "nise_Genuine", "nise_Counterfeit",
    "panadol_Genuine", "panadol_Counterfeit",
    "Background_Unknown"
]

def bundle_complete_dataset():
    print("=========================================================")
    print("      VERICURE: FINAL DATASET PACKAGING PIPELINE         ")
    print("=========================================================\n")

    # 1. Structural Sanity Verification Check
    print("[*] Verifying structural integrity of augmented matrices...")
    missing_classes = []
    for c in EXPECTED_CLASSES:
        class_path = os.path.join(SOURCE_DIR, c)
        if not os.path.exists(class_path):
            missing_classes.append(c)
        else:
            file_count = len([f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            print(f"    -> Found {c}: {file_count} images mapped.")

    if missing_classes:
        print(f"\n[!] Warning: Missing classes detected: {missing_classes}")
        print("[!] Please ensure all augmentation phases have been executed.")
        return

    # 2. Re-map to the official clean production directory name
    if os.path.exists(STAGING_DIR):
        shutil.rmtree(STAGING_DIR)

    print(f"\n[*] Staging directory for renaming: '{STAGING_DIR}'...")
    shutil.copytree(SOURCE_DIR, STAGING_DIR)

    # 3. Compress into the target archive name
    print(f"[*] Compressing '{STAGING_DIR}' into a single size-optimized archive...")
    # archive_format='zip' automatically appends the '.zip' extension to the filename
    archive_path = shutil.make_archive(STAGING_DIR, 'zip', STAGING_DIR)
    print(f"[+] Archive compiled successfully: {os.path.basename(archive_path)}")

    # 4. Export archive directly into your mounted Google Drive directory
    if not os.path.exists(DRIVE_DESTINATION):
        print(f"[!] Target Google Drive path not detected: {DRIVE_DESTINATION}")
        print("[!] Please check that your Drive connection cell is mounted.")
        return

    final_drive_target = os.path.join(DRIVE_DESTINATION, os.path.basename(archive_path))
    print(f"[*] Exporting archive to permanent cloud storage target...")
    shutil.copy(archive_path, final_drive_target)

    print("\n=======================================================================")
    print(f"[SUCCESS] Complete 9-class dataset safely backed up to Google Drive!")
    print(f"LOCATION: VeriCure_Project/dataset_complete.zip")
    print("=======================================================================")

if __name__ == "__main__":
    bundle_complete_dataset()

      VERICURE: FINAL DATASET PACKAGING PIPELINE         

[*] Verifying structural integrity of augmented matrices...
    -> Found froben_Genuine: 500 images mapped.
    -> Found froben_Counterfeit: 500 images mapped.
    -> Found glucophage_Genuine: 500 images mapped.
    -> Found glucophage_Counterfeit: 500 images mapped.
    -> Found nise_Genuine: 500 images mapped.
    -> Found nise_Counterfeit: 500 images mapped.
    -> Found panadol_Genuine: 500 images mapped.
    -> Found panadol_Counterfeit: 500 images mapped.
    -> Found Background_Unknown: 500 images mapped.

[*] Staging directory for renaming: 'dataset_complete'...
[*] Compressing 'dataset_complete' into a single size-optimized archive...
[+] Archive compiled successfully: dataset_complete.zip
[*] Exporting archive to permanent cloud storage target...

[SUCCESS] Complete 9-class dataset safely backed up to Google Drive!
LOCATION: VeriCure_Project/dataset_complete.zip


In [20]:
import zipfile
from collections import Counter
import os

zip_path = "/content/drive/My Drive/VeriCure_Project/dataset_complete.zip"

if not os.path.exists(zip_path):
    print(f"[!] File not found at: {zip_path}")
else:
    print("=========================================================")
    print("      VERICURE: DATASET INTEGRITY REPORT                 ")
    print("=========================================================\n")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Get the list of all file paths inside the zip archive
        all_files = zip_ref.namelist()

        # Extract the top-level directory/class names
        class_counts = Counter()
        for file in all_files:
            # Filter for images to avoid counting directory markers
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                parts = file.split('/')
                if len(parts) > 1:
                    class_name = parts[0]  # If files are zipped inside a root folder
                    if class_name == "dataset_complete" and len(parts) > 2:
                        class_name = parts[1]
                    class_counts[class_name] += 1

        # Print the results in a scannable format
        total_images = 0
        print(f"{'Class Directory Name':<30} | {'Image Count':<12}")
        print("-" * 45)
        for class_name, count in sorted(class_counts.items()):
            print(f"{class_name:<30} | {count:<12}")
            total_images += count

        print("-" * 45)
        print(f"{'TOTAL IMAGES IN ARCHIVE':<30} | {total_images:<12}")
        print("=========================================================")

      VERICURE: DATASET INTEGRITY REPORT                 

Class Directory Name           | Image Count 
---------------------------------------------
Background_Unknown             | 500         
froben_Counterfeit             | 500         
froben_Genuine                 | 500         
glucophage_Counterfeit         | 500         
glucophage_Genuine             | 500         
nise_Counterfeit               | 500         
nise_Genuine                   | 500         
panadol_Counterfeit            | 500         
panadol_Genuine                | 500         
---------------------------------------------
TOTAL IMAGES IN ARCHIVE        | 4500        
